> <p><small>This notebook is made available subject to the licence and terms set out in <a href="https://creativecommons.org/licenses/by/4.0">https://creativecommons.org/licenses/by/4.0</a>.</small></p>

<img src="https://pub-bba109a9a6ac49e3b428cdca19c34363.r2.dev/LT%20-%20Session%204.jpg">

# 4.6 AI Activity Long 1A - Lab: From Embeddings To Attention (Teacher)

Explore the core mechanism of transformer models, self-attention, concretely and intuitively.

60 minutes

## Overview

Attention is not a mysterious black box. It is a pipeline:

1. Tokens start as embeddings.
2. Embeddings are projected into queries, keys, and values.
3. Queries are compared with keys using dot products.
4. Scores are scaled.
5. Softmax converts scores into attention weights.
6. Attention weights combine values into new representations.

The teacher version includes full solutions, visualizations, and notes to support classroom explanation.

---
> ℹ️ **Info:**  
> The vectors and projection matrices are deliberately tiny. The goal is conceptual clarity rather than a production-scale transformer implementation.
---

### What you'll learn:

- Explain attention as weighted aggregation.
- Explain masking as an information constraint.
- Develop intuition for multi‑head attention.


## Student tasks

Use this notebook to support students as they complete the student version.

Students should be able to:

1. Construct queries, keys, and values from token embeddings.
2. Compute attention scores using dot products.
3. Apply scaling and softmax.
4. Visualize attention weights as a heatmap.
5. Apply causal masking.
6. Compare two simple attention heads.

---
> 📝 **Teacher notes:**  
> Encourage students to explain each matrix operation verbally. The key learning goal is understanding the computational sequence.

## Step 1 — Start from token embeddings

Students work with a short 4-token sequence:

- **the**
- **smart**
- **student**
- **studies**

Each token is represented by a small vector. In real transformers, these vectors would be learned embeddings. Here, hand-designed vectors keep every step readable.

In [ ]:
# Import libraries and define a tiny sequence of token embeddings.
import numpy as np
import matplotlib.pyplot as plt

tokens = ["the", "smart", "student", "studies"]

X = np.array([
    [1, 0, 1],
    [1, 2, 0],
    [2, 1, 1],
    [0, 1, 2],
], dtype=float)

X

## Step 2 — Build queries, keys, and values

Attention does not usually compare embeddings directly. Instead, each embedding is projected into three spaces:

- **Query (Q):** what this token is looking for.
- **Key (K):** what this token offers to be matched against.
- **Value (V):** what information this token contributes if selected.

To keep the arithmetic simple, this first head uses identity matrices.

In [ ]:
# Define simple projection matrices and compute Q, K, and V.
W_Q = np.eye(3)
W_K = np.eye(3)
W_V = np.eye(3)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

Q, K, V

## Step 3 — Compute similarity scores

Each token's query vector is compared against every token's key vector:

```
[
  text{scores} = QK^T
]
```

Each entry in the matrix tells students how strongly one token matches another.

In [ ]:
# Compute pairwise dot-product similarity scores.
scores = Q @ K.T
scores

## Step 4 — Scale the scores

Dot-product scores are scaled by:
```
[
  frac{1}{\sqrt{d_k}}
]
```
where `(d_k)` is the dimensionality of the Key vectors.

---
> 📝 **Teacher notes:**  
> Students do not need a full optimization explanation. The key message is that scaling keeps softmax from becoming too sharp when vectors are high-dimensional.

In [ ]:
# Scale scores by the square root of the key dimension.
d_k = K.shape[1]
scaled_scores = scores / np.sqrt(d_k)

scaled_scores

## Step 5 — Convert scores into attention weights with softmax

Softmax converts each row of scores into a probability distribution.

After softmax:

- Weights are non-negative.
- Each row sums to 1.
- Each row describes how much one token attends to all tokens.

In [ ]:
def softmax(x):
    """Compute row-wise softmax for a matrix of scores.

    Args:
        x: A 2D NumPy array of attention scores.

    Returns:
        A 2D NumPy array where each row sums to 1.
    """
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


weights = softmax(scaled_scores)

weights

## Step 6 — Visualize attention weights

A heatmap makes attention easier to interpret than raw numbers.

---
> 📝 **Teacher notes:**  
> This addresses the activity guidance to use visualizations. Ask students to identify the strongest relationship in each row.

In [ ]:
def plot_attention(weights, tokens, title):
    """Display an attention matrix as a heatmap.

    Args:
        weights: A 2D NumPy array of attention weights.
        tokens: Token labels used on both axes.
        title: Title shown above the heatmap.
    """
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(weights)

    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_yticklabels(tokens)

    ax.set_xlabel("Attended-to token")
    ax.set_ylabel("Attending token")
    ax.set_title(title)

    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center")

    fig.colorbar(im, ax=ax)
    plt.show()


plot_attention(weights, tokens, "Head 1 attention weights")

## Step 7 — Compute the new representations

The attention weights are used to combine the value vectors:

```
[
  text{output} = \text{weights} \times V
]
```

This shows attention as weighted aggregation.

In [ ]:
# Combine value vectors using attention weights.
output = weights @ V

output

## Step 8 — Add causal masking

For language generation, a token should not look ahead to future tokens.

Before applying softmax, forbidden future positions are set to a very large negative value. After softmax, these positions receive weight 0.

In [ ]:
# Create and apply a causal mask.
mask = np.triu(np.ones_like(scores), k=1)

masked_scores = scaled_scores.copy()
masked_scores[mask == 1] = -1e9

masked_weights = softmax(masked_scores)
masked_output = masked_weights @ V

masked_weights, masked_output

In [ ]:
# Visualize the effect of causal masking.
plot_attention(masked_weights, tokens, "Causal masked attention weights")

## Step 9 — Add a second attention head

A transformer does not rely on a single attention pattern. It uses multiple heads.

Each head has its own projection matrices, so each one can focus on different relationships. This simple second head keeps the first two embedding dimensions and removes the third.

In [ ]:
# Define a second attention head with different projections.
W_Q2 = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
], dtype=float)

W_K2 = W_Q2.copy()
W_V2 = np.eye(3)

Q2 = X @ W_Q2
K2 = X @ W_K2
V2 = X @ W_V2

scores2 = Q2 @ K2.T
scaled_scores2 = scores2 / np.sqrt(d_k)
weights2 = softmax(scaled_scores2)
output2 = weights2 @ V2

weights2, output2

In [ ]:
# Visualize the second head.
plot_attention(weights2, tokens, "Head 2 attention weights")

## Step 10 — Compare the heads

The purpose of this step is not to claim that one head is correct and the other is wrong.

Students should observe that:

- Changing projections changes scores.
- Changing scores changes weights.
- Changing weights changes output representations.

---
> 📝 **Teacher notes:**  
>  Ask students why several different attention patterns might be useful in a larger transformer.

In [ ]:
# Compare Head 1 and Head 2 numerically.
print("Head 1 weights:")
print(np.round(weights, 2))

print("\nHead 2 weights:")
print(np.round(weights2, 2))

> 💭 **Reflection:**
> Students should now be able to explain this chain:
> 1. Tokens start as embeddings.
> 2. Embeddings are projected into queries, keys, and values.
> 3. Queries are compared to keys.
> 4. Scores are scaled.
> 5. Softmax turns scores into attention weights.
> 6. Weights aggregate values.
> 7. Masking restricts access to future tokens.
> 8. Multiple heads allow different patterns of focus.

> **Suggested closing question:** Which part of the attention pipeline most changes the final representation: the projections, the scores, the softmax, or the mask?